In [33]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential

import warnings
warnings.filterwarnings('ignore')

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [34]:
df = pd.read_csv("/kaggle/input/datasets/ashura369/sentiment-analysis/sentiment-analysis.csv")

In [35]:
print(df.isna().sum())

df = df.dropna()
print(df.isna().sum())

Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score    2
dtype: int64
Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score    0
dtype: int64


In [36]:
data = df['Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score'].str.split(',', expand=True)
data.columns = ['text', 'sentiment', 'source', 'date/time', 'ID', 'location', 'confidence_score']
data.head(10)

,text,sentiment,source,date/time,ID,location,confidence_score
0,"""I love this product!""",Positive,Twitter,2023-06-15 09:23:14,@user123,New York,0.85
1,"""The service was terrible.""",Negative,Yelp Reviews,2023-06-15 11:45:32,user456,Los Angeles,0.65
2,"""This movie is amazing!""",Positive,IMDb,2023-06-15 14:10:22,moviefan789,London,0.92
3,"""I'm so disappointed with their customer suppo...",Negative,Online Forum,2023-06-15 17:35:11,forumuser1,Toronto,0.78
4,"""Just had the best meal of my life!""",Positive,TripAdvisor,2023-06-16 08:50:59,foodie22,Paris,0.88
5,"""The quality of this product is subpar.""",Negative,Amazon Reviews,2023-06-16 10:15:27,shopper123,San Francisco,0.72
6,"""I can't stop listening to this song. It's inc...",Positive,Spotify,2023-06-16 13:40:18,musiclover456,Berlin,0.91
7,"""Their website is so user-friendly. Love it!""",Positive,Website Testimonial,2023-06-16 16:05:36,testimonialuser1,Sydney,0.87
8,"""I loved the movie! It was fantastic!""",Positive,IMDb,2023-07-02 09:12:34,user123,New York,0.92
9,"""The customer service was terrible.""",Negative,Yelp Reviews,2023-07-02 10:45:21,user456,Los Angeles,0.65


In [37]:
data['confidence_score'] = data['confidence_score'].astype(float)
data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   text              96 non-null     object
 1   sentiment         96 non-null     object
 2   source            96 non-null     object
 3   date/time         96 non-null     object
 4   ID                96 non-null     object
 5   location          96 non-null     object
 6   confidence_score  96 non-null     float64
dtypes: float64(1), object(6)
memory usage: 15.2+ KB


In [38]:
print(data.shape)

data = data.drop_duplicates(subset=['ID'])
print(data.shape)

(96, 7)
(73, 7)


In [39]:
data['text'] = data['text'].str.replace('"','').str.strip()
data['sentiment'] = data['sentiment'].str.strip()


print("Sentiments count: ", data['sentiment'].value_counts())

Sentiments count:  sentiment
Positive    42
Negative    31
Name: count, dtype: int64


In [40]:
data['target'] = data['sentiment'].map({
    'Positive':1,
    'Negative':0
})

data['target'] = data['target'].astype(int)

In [41]:
data[['text','sentiment','target']].head(10)

,text,sentiment,target
0,I love this product!,Positive,1
1,The service was terrible.,Negative,0
2,This movie is amazing!,Positive,1
3,I'm so disappointed with their customer support.,Negative,0
4,Just had the best meal of my life!,Positive,1
5,The quality of this product is subpar.,Negative,0
6,I can't stop listening to this song. It's incr...,Positive,1
7,Their website is so user-friendly. Love it!,Positive,1
8,I loved the movie! It was fantastic!,Positive,1
10,This book made me feel inspired. Highly recomm...,Positive,1


In [42]:
x = data['text'].copy()
y = data['target'].copy()

In [43]:
x

0                                  I love this product!
1                             The service was terrible.
2                                This movie is amazing!
3      I'm so disappointed with their customer support.
4                    Just had the best meal of my life!
                            ...                        
69    The website navigation is smooth and intuitive...
70    The ride on this cruise ship was an unforgetta...
71    The product I received was of poor quality. It...
72    This book kept me hooked from start to finish....
73    The customer service at this store is outstand...
Name: text, Length: 73, dtype: object

In [44]:
def transform(x):
    max_vocab_size = 1000
    max_sequence_len = 25
    
    vectorizer = layers.TextVectorization(
        max_tokens = max_vocab_size,
        output_sequence_length = max_sequence_len,
        output_mode = 'int'
    )
    
    # converting x to a standard CPU numpy array of strings (coz cudf based data['text'] of pandas is not supported here)
    x = data['text'].astype(str).to_numpy()
    
    # learning vocabulary from dataset
    vectorizer.adapt(x)
    
    # transforming text to integer sequence matrices
    x = vectorizer(x)

    return x


x = transform(x)

In [46]:
print(x.shape)
print()
print(len(vectorizer.get_vocabulary()))
print()
print(x)
print()
print(vectorizer.get_vocabulary())

(73, 25)

274

tf.Tensor(
[[  4  84   3 ...   0   0   0]
 [  2  17   6 ...   0   0   0]
 [  3  35   5 ...   0   0   0]
 ...
 [  2  15   4 ...   0   0   0]
 [  3  34 190 ...   0   0   0]
 [  2  14  17 ...   0   0   0]], shape=(73, 25), dtype=int64)

['', '[UNK]', np.str_('the'), np.str_('this'), np.str_('i'), np.str_('is'), np.str_('was'), np.str_('a'), np.str_('and'), np.str_('at'), np.str_('to'), np.str_('with'), np.str_('their'), np.str_('of'), np.str_('customer'), np.str_('product'), np.str_('im'), np.str_('service'), np.str_('it'), np.str_('had'), np.str_('website'), np.str_('food'), np.str_('experience'), np.str_('disappointed'), np.str_('terrible'), np.str_('restaurant'), np.str_('support'), np.str_('my'), np.str_('me'), np.str_('song'), np.str_('so'), np.str_('quality'), np.str_('its'), np.str_('cant'), np.str_('book'), np.str_('movie'), np.str_('highly'), np.str_('arrived'), np.str_('store'), np.str_('roller'), np.str_('ride'), np.str_('recommended'), np.str_('poor'), np.str_('

In [53]:
x = np.array(x)
y = np.array(y)

print(type(x))
print(type(y))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [54]:
from sklearn.model_selection import train_test_split as tts

x_train, x_test, y_train, y_test = tts(x,y, test_size=0.2, random_state=42)

print(x_train.shape)
print(x_test.shape)

(58, 25)
(15, 25)


### Simple RNN model

In [55]:
print(x.shape)
print()
print(len(vectorizer.get_vocabulary()))
print()
print(x)

(73, 25)

274

[[  4  84   3 ...   0   0   0]
 [  2  17   6 ...   0   0   0]
 [  3  35   5 ...   0   0   0]
 ...
 [  2  15   4 ...   0   0   0]
 [  3  34 190 ...   0   0   0]
 [  2  14  17 ...   0   0   0]]


In [69]:
vocab_size = len(vectorizer.get_vocabulary())
embedding_dimenstion = 32

model = Sequential([
    layers.Input(shape=(25,)),
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dimenstion),
    layers.LSTM(64, return_sequences=True, dropout=0.2),
    layers.Dropout(0.2),
    layers.LSTM(32, dropout=0.2),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss = keras.losses.BinaryCrossentropy(),
    optimizer = keras.optimizers.Adam(),
    metrics = ['accuracy']
)

history = model.fit(
    x_train, y_train,
    validation_data = (x_test, y_test),
    epochs = 10,
    batch_size = 64
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.5690 - loss: 0.6925 - val_accuracy: 0.6667 - val_loss: 0.6892
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.5517 - loss: 0.6909 - val_accuracy: 0.6667 - val_loss: 0.6855
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.5690 - loss: 0.6907 - val_accuracy: 0.6667 - val_loss: 0.6820
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5517 - loss: 0.6864 - val_accuracy: 0.6667 - val_loss: 0.6785
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5517 - loss: 0.6946 - val_accuracy: 0.6667 - val_loss: 0.6756
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.5517 - loss: 0.6865 - val_accuracy: 0.6667 - val_loss: 0.6727
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5517 - loss: 0.6860 - val_accuracy: 0.6667 - val_loss: 0.6697
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5517 - loss: 0.6841 - val_accuracy: 0.6667 - val_loss: 0.

In [70]:
pred = model.predict(x_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step


In [71]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6667 - loss: 0.6625
Test Accuracy: 66.67%


In [72]:
from sklearn.metrics import accuracy_score


y_pred = (pred > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {acc * 100:.2f}%")

Accuracy Score: 66.67%


In [73]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Confusion Matrix:
[[ 0  5]
 [ 0 10]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         5
    Positive       0.67      1.00      0.80        10

    accuracy                           0.67        15
   macro avg       0.33      0.50      0.40        15
weighted avg       0.44      0.67      0.53        15



### Optuna based code

In [82]:
import optuna
from tensorflow.keras import regularizers

In [86]:

vocab_size = len(vectorizer.get_vocabulary())

def my_model(trial):
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1, 3)
    
    embedding_dim = trial.suggest_int('embedding_dim', 16, 64, step=16)
    embedding_initializer = trial.suggest_categorical('embeddings_initializer', ['glorot_normal', 'he_normal'])

    model = keras.Sequential([
        layers.Input(shape=(25,)),
        layers.Embedding(
            input_dim=vocab_size, 
            output_dim=embedding_dim, 
            embeddings_initializer=embedding_initializer
        )
    ])

    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_{i}', 16, 128, step=16)
        dropout_rate = trial.suggest_float(f'dropout_{i}', 0.0, 0.5, step=0.1)
        
        return_sequences = True if i < (n_layers - 1) else False

        model.add(layers.LSTM(neurons, return_sequences=return_sequences))
        
        if dropout_rate > 0.0:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        loss=keras.losses.BinaryCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        metrics=['accuracy']
    )

    # 5. Train Model
    history = model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=6,
        batch_size=64,
        verbose=1
    )

    model.save(f"model_trial_{trial.number}.keras")

    return max(history.history['val_accuracy'])


study = optuna.create_study(direction='maximize')
study.optimize(my_model, n_trials=10)

print("Best Parameters:", study.best_params)

[I 2026-09-29 02:52:57,403] A new study created in memory with name: no-name-81dc4bb3-aa0b-43ea-b22c-9fd4531545eb


Epoch 1/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6939 - val_accuracy: 0.6667 - val_loss: 0.6905
Epoch 2/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.5517 - loss: 0.6930 - val_accuracy: 0.6667 - val_loss: 0.6891
Epoch 3/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.5172 - loss: 0.6918 - val_accuracy: 0.6667 - val_loss: 0.6879
Epoch 4/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.5000 - loss: 0.6921 - val_accuracy: 0.6667 - val_loss: 0.6867
Epoch 5/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5517 - loss: 0.6881 - val_accuracy: 0.6667 - val_loss: 0.6855
Epoch 6/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.5172 - loss: 0.6916 - val_accuracy: 0.6667 - val_loss: 0.6844


[I 2026-09-29 02:52:59,822] Trial 0 finished with value: 0.6666666865348816 and parameters: {'lr': 0.00031779737740006593, 'n_layers': 1, 'embedding_dim': 16, 'embeddings_initializer': 'he_normal', 'neurons_0': 48, 'dropout_0': 0.2}. Best is trial 0 with value: 0.6666666865348816.


Epoch 1/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5172 - loss: 0.6924 - val_accuracy: 0.6667 - val_loss: 0.6799
Epoch 2/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.5517 - loss: 0.6924 - val_accuracy: 0.6667 - val_loss: 0.6716
Epoch 3/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.5517 - loss: 0.6913 - val_accuracy: 0.6667 - val_loss: 0.6656
Epoch 4/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.5517 - loss: 0.6857 - val_accuracy: 0.6667 - val_loss: 0.6596
Epoch 5/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.5517 - loss: 0.6840 - val_accuracy: 0.6667 - val_loss: 0.6537
Epoch 6/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.5517 - loss: 0.6812 - val_accuracy: 0.6667 - val_loss: 0.6477


[I 2026-09-29 02:53:03,737] Trial 1 finished with value: 0.6666666865348816 and parameters: {'lr': 0.0022931780860158987, 'n_layers': 3, 'embedding_dim': 32, 'embeddings_initializer': 'glorot_normal', 'neurons_0': 48, 'dropout_0': 0.1, 'neurons_1': 32, 'dropout_1': 0.30000000000000004, 'neurons_2': 48, 'dropout_2': 0.5}. Best is trial 0 with value: 0.6666666865348816.


...

Epoch 1/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.4483 - loss: 0.6954 - val_accuracy: 0.6667 - val_loss: 0.6703
Epoch 2/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5517 - loss: 0.6884 - val_accuracy: 0.6667 - val_loss: 0.6539
Epoch 3/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.5517 - loss: 0.6897 - val_accuracy: 0.6667 - val_loss: 0.6516
Epoch 4/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.5517 - loss: 0.6901 - val_accuracy: 0.6667 - val_loss: 0.6548
Epoch 5/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.5517 - loss: 0.6886 - val_accuracy: 0.6667 - val_loss: 0.6599
Epoch 6/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.5517 - loss: 0.6874 - val_accuracy: 0.6667 - val_loss: 0.6651


[I 2026-09-29 02:53:24,946] Trial 8 finished with value: 0.6666666865348816 and parameters: {'lr': 0.0013219499030506913, 'n_layers': 1, 'embedding_dim': 64, 'embeddings_initializer': 'he_normal', 'neurons_0': 128, 'dropout_0': 0.0}. Best is trial 4 with value: 0.800000011920929.


Epoch 1/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.3966 - loss: 0.6950 - val_accuracy: 0.3333 - val_loss: 0.6943
Epoch 2/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.4483 - loss: 0.6936 - val_accuracy: 0.2667 - val_loss: 0.6937
Epoch 3/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.5000 - loss: 0.6930 - val_accuracy: 0.6000 - val_loss: 0.6932
Epoch 4/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.5000 - loss: 0.6929 - val_accuracy: 0.6000 - val_loss: 0.6927
Epoch 5/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.5690 - loss: 0.6919 - val_accuracy: 0.6000 - val_loss: 0.6921
Epoch 6/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.5690 - loss: 0.6925 - val_accuracy: 0.6667 - val_loss: 0.6916


[I 2026-09-29 02:53:29,695] Trial 9 finished with value: 0.6666666865348816 and parameters: {'lr': 4.962241875442245e-05, 'n_layers': 3, 'embedding_dim': 64, 'embeddings_initializer': 'he_normal', 'neurons_0': 64, 'dropout_0': 0.1, 'neurons_1': 112, 'dropout_1': 0.30000000000000004, 'neurons_2': 112, 'dropout_2': 0.30000000000000004}. Best is trial 4 with value: 0.800000011920929.


Best Parameters: {'lr': 0.006406953407197273, 'n_layers': 3, 'embedding_dim': 64, 'embeddings_initializer': 'glorot_normal', 'neurons_0': 112, 'dropout_0': 0.1, 'neurons_1': 16, 'dropout_1': 0.30000000000000004, 'neurons_2': 112, 'dropout_2': 0.4}


In [87]:
print(study.best_trial.number)
print()
study.best_trial

4



FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[0.800000011920929], datetime_start=datetime.datetime(2026, 9, 29, 2, 53, 10, 832258), datetime_complete=datetime.datetime(2026, 9, 29, 2, 53, 14, 876663), params={'lr': 0.006406953407197273, 'n_layers': 3, 'embedding_dim': 64, 'embeddings_initializer': 'glorot_normal', 'neurons_0': 112, 'dropout_0': 0.1, 'neurons_1': 16, 'dropout_1': 0.30000000000000004, 'neurons_2': 112, 'dropout_2': 0.4}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.01, log=True, low=1e-05, step=None), 'n_layers': IntDistribution(high=3, log=False, low=1, step=1), 'embedding_dim': IntDistribution(high=64, log=False, low=16, step=16), 'embeddings_initializer': CategoricalDistribution(choices=('glorot_normal', 'he_normal')), 'neurons_0': IntDistribution(high=128, log=False, low=16, step=16), 'dropout_0': FloatDistribution(high=0.5, log=False, low=0.0, step=0.1), 'neurons_1': IntDistribution(high=128, l

In [88]:
num = study.best_trial.number
model = keras.models.load_model(f"model_trial_{num}.keras")
model.summary()

Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_21 (Embedding)        │ (None, 25, 64)         │        17,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_38 (LSTM)                  │ (None, 25, 112)        │        79,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 25, 112)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_39 (LSTM)                  │ (None, 25, 16)         │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_35 (Dropout)            │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_40 (LSTM)                  │ (None, 112)            │        57,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │           113 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 488,981 (1.87 MB)

 Trainable params: 162,993 (636.69 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 325,988 (1.24 MB)

In [89]:
model.fit(
    x_train, y_train,
    epochs = 20,
    batch_size=256
)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.8276 - loss: 0.3401
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.8966 - loss: 0.2392
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 1.0000 - loss: 0.0761
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.9655 - loss: 0.1052
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 1.0000 - loss: 0.0186
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 1.0000 - loss: 0.0069
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 1.0000 - loss: 0.0038
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 1.0000 - loss: 0.0032
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9828 - loss: 0.0654
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 1.0000 - loss: 0.0014
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 1.0000 - loss: 0.0141
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.9828 - loss: 0.0321
Epo

In [90]:
pred = model.predict(x_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step


In [91]:
predicted_classes = (pred >= 0.5).astype(int).flatten()

accuracy = np.mean(predicted_classes == y_test)
print(f"Corrected Test Accuracy: {accuracy * 100:.2f}%\n")

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(y_test, predicted_classes))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predicted_classes))

Corrected Test Accuracy: 60.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.44      0.80      0.57         5
           1       0.83      0.50      0.62        10

    accuracy                           0.60        15
   macro avg       0.64      0.65      0.60        15
weighted avg       0.70      0.60      0.61        15


Confusion Matrix:
[[4 1]
 [5 5]]
